## Instalación de dependencias

In [ ]:
!pip install requests pandas

## Imports y configuración

In [ ]:
import requests
import pandas as pd
import time
import os
import re
import json

# ── Credenciales ──────────────────────────────────────────────────────────────
API_KEY = ""## Clave de acceso personal a la API. Por motivos de seguridad, la utilizada en este estudio ha sido eliminada.
URL     = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEADERS = {"X-API-Key": API_KEY}

# ── Páginas por defecto y por mes de alta actividad ───────────────────────────
# Basado en el análisis del log v2:
# Meses con más volumen histórico: sep(9), mar(3), feb(2), nov(11), ene(1)
# → se les asigna MAX_PAGES_ALTO; el resto usa MAX_PAGES_BASE
MAX_PAGES_BASE  = 5   # meses de actividad normal
MAX_PAGES_ALTO  = 8   # meses de alta actividad académica

MESES_ALTO_VOLUMEN = {1, 2, 3, 9, 11}   # enero, febrero, marzo, septiembre, noviembre

# ── Pausas ────────────────────────────────────────────────────────────────────
SLEEP_BETWEEN_REQUESTS = 1.5
SLEEP_BETWEEN_MONTHS   = 3

# ── Rutas de salida ────────────────────────────────────────────────────────────
OUT_DIR    = "data/uam_v3_monthly"
FINAL_PATH = "data/uam_tweets_v3.csv"
LOG_PATH   = "data/uam_v3_log.json"

os.makedirs(OUT_DIR, exist_ok=True)
print("Configuración lista.")
print(f"  MAX_PAGES base:        {MAX_PAGES_BASE}")
print(f"  MAX_PAGES alto volumen: {MAX_PAGES_ALTO}  (meses: {sorted(MESES_ALTO_VOLUMEN)})")


## Lista de exclusión de cuentas institucionales




In [ ]:
CUENTAS_EXCLUIR = {
    # Cuenta oficial UAM Madrid y subcuentas
    'UAM_Madrid', 'fuam_uam', 'UCCUAM', 'meaic_uam', 'CBM_CSIC_UAM',
    'EDoctorado_UAM', 'EUECREM_UAM', 'UAM_Biblioteca', 'EdinnovaUAM',
    'VEmerg_ExperUAM', 'GeografiaUam', 'AnaLopezUAM', 'Contemporan_UAM',
    'HistoriaUam', 'FyL_UAM', 'derecho_uam', 'CBMSO_CSIC_UAM',
    # Cuentas UAM México detectadas en v2
    'uamxoficial', 'UAMIztapalapa', 'UAMAzcapotzalco', 'UAMXochimilco',
    'UAMLerma', 'UAMCuajimalpa',
    # Medios de comunicación
    'isanidad', '24horas_rne', 'ondaceromn', 'SERMadridNorte',
    'Conversation_E', 'elpaiseducacion',
    # Entidades científicas y fundaciones
    'FundacionLilly', 'fundacionsi', 'ENAIRE', 'ONCE_oficial',
    'LRicoti', 'EnsedeCiencia', 'FECYT_Ciencia',
    'EventoCiencia', 'UniCienciaSomos', 'RACiencias',
    # Oposiciones / academias
    'OpositaTest', 'Registrador_es',
    # Otros no estudiantiles
    'MusicBAcademy', 'Hospital_FJD', 'CopMadrid',
}

print(f"Cuentas en lista de exclusión: {len(CUENTAS_EXCLUIR)}")


## Filtro de desambiguación UAM Madrid / UAM México




In [ ]:
KW_MEXICO = [
    'uam xochimilco', 'uam iztapalapa', 'uam azcapotzalco',
    'uam lerma', 'uam cuajimalpa', 'correo.uam.mx', 'uam.mx',
    'casa abierta al tiempo', 'cdmx', 'ciudad de méxico',
    'estado de méxico', 'conacyt', 'unam', 'ipn',
]
PATRON_MEXICO = re.compile('|'.join(re.escape(k) for k in KW_MEXICO), re.IGNORECASE)

def es_tweet_mexico(text):
    return bool(PATRON_MEXICO.search(str(text)))

print(f"Filtro México listo — {len(KW_MEXICO)} términos de exclusión.")


## Función de páginas adaptativas





In [ ]:
def get_max_pages(month_label):
    """
    Devuelve el número de páginas a descargar según el mes.
    Los meses de alta actividad académica usan MAX_PAGES_ALTO;
    el resto usa MAX_PAGES_BASE.

    Parámetro:
        month_label (str): etiqueta del mes, p.ej. '2024_09'
    """
    mes_num = int(month_label.split('_')[1])
    if mes_num in MESES_ALTO_VOLUMEN:
        return MAX_PAGES_ALTO
    return MAX_PAGES_BASE


# Verificación visual
print("Páginas asignadas por mes:")
print(f"{'Mes':>4}  {'Páginas':>7}  {'Motivo'}")
print("-" * 35)
for m in range(1, 13):
    label  = f"2024_{m:02d}"
    pages  = get_max_pages(label)
    motivo = "alto volumen" if m in MESES_ALTO_VOLUMEN else "base"
    marker = " ◀" if m in MESES_ALTO_VOLUMEN else ""
    print(f"  {m:02d}    {pages:>7}  {motivo}{marker}")

# Estimación de créditos totales para 39 meses
meses_alto = sum(1 for _, s, _ in
                 [(f"2023_{m:02d}", None, None) for m in range(1,13)] +
                 [(f"2024_{m:02d}", None, None) for m in range(1,13)] +
                 [(f"2025_{m:02d}", None, None) for m in range(1,13)] +
                 [(f"2026_{m:02d}", None, None) for m in range(1,4)]
                 if int(_.split('_')[1] if _ else s or "0_01") in MESES_ALTO_VOLUMEN
                 ) if False else sum(
    1 for m in list(range(1,13))*3 + list(range(1,4))
    if m in MESES_ALTO_VOLUMEN
)
meses_base = 39 - meses_alto
llamadas_est = (meses_alto * MAX_PAGES_ALTO + meses_base * MAX_PAGES_BASE) * 3
print(f"\nEstimación de créditos para 39 meses:")
print(f"  Meses alto volumen: {meses_alto} × {MAX_PAGES_ALTO} páginas × 3 queries = {meses_alto * MAX_PAGES_ALTO * 3} llamadas")
print(f"  Meses base:         {meses_base} × {MAX_PAGES_BASE} páginas × 3 queries = {meses_base * MAX_PAGES_BASE * 3} llamadas")
print(f"  TOTAL estimado:     {llamadas_est} llamadas")


## Generador de rangos mensuales

In [ ]:
def generate_month_ranges(start_year=2023, start_month=1,
                          end_year=2026,   end_month=3):
    ranges = []
    year, month = start_year, start_month
    while (year < end_year) or (year == end_year and month <= end_month):
        start_date = f"{year:04d}-{month:02d}-01"
        if month == 12:
            next_year, next_month = year + 1, 1
        else:
            next_year, next_month = year, month + 1
        end_date = f"{next_year:04d}-{next_month:02d}-01"
        ranges.append((f"{year:04d}_{month:02d}", start_date, end_date))
        year, month = next_year, next_month
    return ranges

month_ranges = generate_month_ranges()
print(f"Total meses: {len(month_ranges)}  |  {month_ranges[0][0]} → {month_ranges[-1][0]}")


## Queries v3




In [ ]:
def build_queries_v3(start_date, end_date):

    ancla_madrid = (
        '(@UAM_Madrid OR "Autónoma de Madrid" OR "Universidad Autónoma de Madrid" '
        'OR cantoblanco OR "UAM Madrid")'
    )

    queries = {

        # ── Q1: experiencia directa ───────────────────────────────────────────
        # La ancla geográfica está integrada en el vocabulario: "en la autónoma",
        # "en cantoblanco", "en la UAM Madrid" son suficientemente específicos.
        "experiencia_uam_madrid": (
            f'("en la autónoma" OR "en cantoblanco" OR "en la UAM Madrid" '
            f'OR "años en la UAM" OR "años en la Autónoma" '
            f'OR "estudio en la UAM" OR "estudié en la UAM" '
            f'OR "estudié en la Autónoma" OR "soy de la Autónoma" '
            f'OR "mi facultad en la UAM" OR "mi carrera en la UAM" '
            f'OR "mi tfg en la UAM" OR "mi tfm en la UAM" '
            f'OR "me han cateado en" OR "suspendí en la UAM" '
            f'OR "aprobé en la UAM" OR "matrícula en la UAM" '
            f'OR "beca en la UAM" OR "erasmus en la UAM" '
            f'OR "secretaría de la UAM" OR "campus de la UAM") '
            f'lang:es since:{start_date} until:{end_date}'
        ),

        # ── Q2: valoración explícita ──────────────────────────────────────────
        "valoracion_uam_madrid": (
            f'{ancla_madrid} '
            f'(recomiendo OR "no recomiendo" OR "ojalá hubiera" '
            f'OR "una mierda" OR "lo mejor" OR "lo peor" '
            f'OR "muy bien" OR "muy mal" OR "muy buena" OR "muy mala" '
            f'OR genial OR horrible OR fatal OR increíble OR decepcionante '
            f'OR satisfecho OR insatisfecho OR "estoy harto" OR encantado '
            f'OR "odio la UAM" OR "amo la UAM" OR vergüenza OR excelente '
            f'OR pésimo OR nefasto OR "me arrepiento" OR "mejor decisión" '
            f'OR "peor decisión") '
            f'lang:es since:{start_date} until:{end_date}'
        ),

        # ── Q3: momentos académicos ───────────────────────────────────────────
        "momentos_academicos_madrid": (
            f'{ancla_madrid} '
            f'(examen OR matrícula OR "nota final" OR suspenso OR aprobado '
            f'OR "trabajo fin de grado" OR "trabajo fin de máster" '
            f'OR tfg OR tfm OR prácticas OR beca OR erasmus '
            f'OR secretaría OR "lista de espera" OR convocatoria '
            f'OR horario OR "nota de corte" OR selectividad OR EBAU '
            f'OR "cambio de grupo" OR "guía docente" OR "acta de notas" '
            f'OR huelga OR manifestación) '
            f'lang:es since:{start_date} until:{end_date}'
        ),
    }
    return queries


# Vista previa
ejemplo = build_queries_v3("2024-09-01", "2024-10-01")
for nombre, q in ejemplo.items():
    print(f"[{nombre}]")
    print(f"  {q[:220]}...")
    print()


## Función de descarga por mes




In [ ]:
def download_month_v3(label, start_date, end_date):
    """
    Descarga todos los tweets de un mes para las 3 queries v3.
    El número de páginas se determina automáticamente según el mes.
    """
    max_pages = get_max_pages(label)
    queries   = build_queries_v3(start_date, end_date)
    all_rows  = []
    stats     = {}

    print(f"\n{'='*52}")
    print(f"  Mes: {label}  ({start_date} → {end_date})  [max_pages={max_pages}]")
    print(f"{'='*52}")

    for query_name, query in queries.items():
        print(f"\n  >> {query_name}")

        next_cursor = None
        page        = 1
        n_api       = 0
        n_inst      = 0
        n_mx        = 0

        while True:
            params = {"query": query, "queryType": "Latest"}
            if next_cursor:
                params["cursor"] = next_cursor

            try:
                resp = requests.get(URL, headers=HEADERS, params=params, timeout=30)
                resp.raise_for_status()
                data = resp.json()
            except requests.exceptions.RequestException as e:
                print(f"    ⚠️  Error en página {page}: {e}. Saltando.")
                break

            tweets = data.get("tweets", [])
            print(f"    Página {page}: {len(tweets)} tweets de la API")
            n_api += len(tweets)

            for t in tweets:
                author   = t.get("author") or {}
                username = author.get("userName", "")
                text     = t.get("text", "")

                # Filtro 1: cuenta institucional
                if username in CUENTAS_EXCLUIR:
                    n_inst += 1
                    continue

                # Filtro 2: contexto México
                if es_tweet_mexico(text):
                    n_mx += 1
                    continue

                all_rows.append({
                    "month"           : label,
                    "tweet_id"        : t.get("id"),
                    "created_at"      : t.get("createdAt"),
                    "text"            : text,
                    "lang"            : t.get("lang"),
                    "likes"           : t.get("likeCount", 0),
                    "replies"         : t.get("replyCount", 0),
                    "retweets"        : t.get("retweetCount", 0),
                    "quotes"          : t.get("quoteCount", 0),
                    "views"           : t.get("viewCount", 0),
                    "author_username" : username,
                    "author_name"     : author.get("name"),
                    "url"             : t.get("url"),
                    "query_type"      : query_name,
                    "query_used"      : query,
                    "page"            : page,
                    "max_pages_usado" : max_pages,
                })

            # Guardado incremental
            if all_rows:
                pd.DataFrame(all_rows).drop_duplicates(subset=["tweet_id"]).to_csv(
                    f"{OUT_DIR}/uam_{label}_progress.csv",
                    index=False, encoding="utf-8-sig"
                )

            has_next    = data.get("has_next_page", False)
            next_cursor = data.get("next_cursor")

            if not has_next or not next_cursor:
                print("    Sin más páginas.")
                break
            if page >= max_pages:
                print(f"    Límite de {max_pages} páginas alcanzado.")
                break

            page += 1
            time.sleep(SLEEP_BETWEEN_REQUESTS)

        n_validos = n_api - n_inst - n_mx
        stats[query_name] = {
            "max_pages_usado"        : max_pages,
            "tweets_api"             : n_api,
            "tweets_descartados_inst": n_inst,
            "tweets_descartados_mx"  : n_mx,
            "tweets_validos"         : n_validos,
        }
        print(f"    ✓ API:{n_api}  Inst:{n_inst}  México:{n_mx}  Válidos:{n_validos}")

    month_df = pd.DataFrame(all_rows).drop_duplicates(subset=["tweet_id"])
    out_path = f"{OUT_DIR}/uam_{label}.csv"
    month_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n  ✅ {label} → {out_path}  ({len(month_df)} tweets únicos)")
    return month_df, stats


## Test con un mes




In [ ]:
# Septiembre → mes de alto volumen → usará MAX_PAGES_ALTO=8 automáticamente
test_df, test_stats = download_month_v3("2024_09", "2024-09-01", "2024-10-01")

print("\n--- Estadísticas del test ---")
for q, s in test_stats.items():
    print(f"  {q}:")
    for k, v in s.items():
        print(f"    {k}: {v}")
print(f"\nTweets únicos recogidos: {len(test_df)}")


In [ ]:
# Inspección de calidad
print("Distribución por query:")
print(test_df["query_type"].value_counts())
print()

# ¿Hay tweets de México que se hayan colado?
from_mx = test_df['text'].apply(es_tweet_mexico).sum()
print(f"Tweets de México que pasaron el filtro: {from_mx}")
print()

print("Muestra por query:")
for qtype in test_df["query_type"].unique():
    muestra = test_df[test_df["query_type"] == qtype]["text"].head(3)
    print(f"\n  [{qtype}]")
    for t in muestra:
        print(f"    → {t[:180]}")


## Descarga completa




In [ ]:
month_ranges = generate_month_ranges(
    start_year=2023, start_month=1,
    end_year=2026,   end_month=3,
)

all_months  = []
log_global  = {}
meses_error = []

for label, start_date, end_date in month_ranges:

    # Saltar si ya está descargado
    out_path = f"{OUT_DIR}/uam_{label}.csv"
    if os.path.exists(out_path):
        print(f"    {label} ya descargado, cargando desde disco...")
        all_months.append(pd.read_csv(out_path))
        continue

    try:
        month_df, stats = download_month_v3(label, start_date, end_date)
        all_months.append(month_df)
        log_global[label] = stats
        time.sleep(SLEEP_BETWEEN_MONTHS)
    except Exception as e:
        print(f"   Error en {label}: {e}")
        meses_error.append(label)

with open(LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(log_global, f, ensure_ascii=False, indent=2)

print(f"\n{'='*50}")
print(f"Descarga completada.")
print(f"Meses con error: {meses_error if meses_error else 'ninguno'}")
print(f"Log guardado en: {LOG_PATH}")


## Consolidación del dataset final

In [ ]:
if all_months:
    df_final = pd.concat(all_months, ignore_index=True)
    df_final = df_final.drop_duplicates(subset=["tweet_id"])
    df_final.to_csv(FINAL_PATH, index=False, encoding="utf-8-sig")

    print(f" Dataset guardado en: {FINAL_PATH}")
    print(f"   Total tweets únicos: {len(df_final)}")
    print(f"\nPor query:")
    print(df_final["query_type"].value_counts())
    print(f"\nPor año-mes:")
    print(df_final["month"].value_counts().sort_index().to_string())
else:
    print("No se descargó ningún mes.")


## Resumen del log de recogida

In [ ]:
if log_global:
    filas = []
    for mes, qs in log_global.items():
        filas.append({
            "mes"         : mes,
            "max_pages"   : list(qs.values())[0]["max_pages_usado"],
            "api"         : sum(q["tweets_api"]               for q in qs.values()),
            "inst"        : sum(q["tweets_descartados_inst"]  for q in qs.values()),
            "mexico"      : sum(q["tweets_descartados_mx"]    for q in qs.values()),
            "validos"     : sum(q["tweets_validos"]           for q in qs.values()),
        })

    df_log = pd.DataFrame(filas).set_index("mes")
    df_log["% descarte"] = (
        (df_log["inst"] + df_log["mexico"]) / df_log["api"].replace(0, float("nan")) * 100
    ).round(1)

    print(df_log.to_string())
    print(f"\nTotales:")
    print(f"  Tweets recibidos de la API:  {df_log['api'].sum()}")
    print(f"  Descartados institucionales: {df_log['inst'].sum()}")
    print(f"  Descartados México:          {df_log['mexico'].sum()}")
    print(f"  Válidos:                     {df_log['validos'].sum()}")


## Combinación con corpus v1 y v2



In [ ]:
PATH_V1 = "data/uam_tweets_2023_2026.csv"
PATH_V2 = "data/uam_tweets_v2.csv"
PATH_V3 = FINAL_PATH

dfs = []
for path, version in [(PATH_V1, "v1"), (PATH_V2, "v2"), (PATH_V3, "v3")]:
    if os.path.exists(path):
        tmp = pd.read_csv(path)
        tmp["version"] = version
        dfs.append(tmp)
        print(f"  {version}: {len(tmp)} tweets cargados")
    else:
        print(f"  {version}: no encontrado en {path}")

if dfs:
    df_combinado = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=["tweet_id"])
    df_combinado.to_csv("data/uam_corpus_completo.csv", index=False, encoding="utf-8-sig")
    print(f"\nCorpus combinado: {len(df_combinado)} tweets únicos")
    print(f"   Guardado en: data/uam_corpus_completo.csv")
    print(f"\nDistribución por versión:")
    print(df_combinado["version"].value_counts())
